# Day 70: Image Classification with CNN

## Loading Image Data
This notebook works with the real MNIST dataset (70,000 handwritten digit images, 28x28 pixels each) via `keras.datasets.mnist`, which is the classic benchmark dataset for image classification. `mnist.load_data()` downloads it the first time you run it (cached locally after that, so later runs work offline) and returns it already split into train and test sets. The code below reshapes each image to add an explicit channel dimension (`28, 28, 1` for grayscale, since Conv2D layers expect a channel axis), converts pixel values to float and divides by 255 to scale them into the 0-1 range (raw pixels are 0-255 integers, and neural networks train better on small, normalized inputs), and one-hot encodes the labels for use with softmax + categorical crossentropy.</cell id="cell-1">


In [ ]:
# pip install tensorflow
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

(X_train, y_train), (X_test, y_test) = mnist.load_data()  # downloads/caches MNIST; pre-split into train/test
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255  # add channel dim + scale pixels to [0, 1]
X_test = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255
y_train_cat = to_categorical(y_train)  # one-hot encode digit labels (0-9) for softmax output
y_test_cat = to_categorical(y_test)

## Building the CNN
This is the same layered pattern introduced in the CNN basics notebook, now applied to real MNIST images: two Conv2D + MaxPooling2D blocks that progressively extract and downsample features (32 filters, then 64 filters, each learning increasingly abstract patterns), followed by a Flatten layer to turn the extracted feature maps into a 1D vector, a Dense(64) layer to combine those features, and a final Dense(10, softmax) layer to output a probability for each of the 10 digit classes. This "compact" architecture (only ~2 conv blocks) is intentionally small so it trains quickly for teaching purposes - production image classifiers typically stack more conv blocks and use techniques like batch normalization and data augmentation for better accuracy. `categorical_crossentropy` is used because the labels are one-hot encoded and there are more than 2 classes.</cell id="cell-3">


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),  # first conv block: 32 filters detect low-level patterns (edges, curves)
    layers.MaxPooling2D((2, 2)),  # downsample to reduce size and computation
    layers.Conv2D(64, (3, 3), activation='relu'),  # second conv block: 64 filters detect higher-level combinations of features
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),  # turn 3D feature maps into a 1D vector for the Dense layers
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')  # probability distribution over the 10 digit classes
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

## Training the Model
Training a CNN on the full 60,000-image MNIST training set can take a while, especially on a CPU, so here we train on just the first 5,000 images to keep this a fast, interactive exercise - in a real project you'd typically use the full dataset (and more epochs) once you're confident the pipeline works. `validation_split=0.1` holds out 10% of that 5,000-image subset to track generalization during training, and we only run 5 epochs since this is a quick demonstration rather than a fully-tuned model. The printed final training accuracy shows how well the model fit the (small) training subset it saw - don't expect this number alone to represent real-world performance, which is what the next cell's test evaluation is for.</cell id="cell-5">


In [ ]:
history = model.fit(X_train[:5000], y_train_cat[:5000], epochs=5, validation_split=0.1, verbose=0)  # train on a 5,000-image subset to keep this fast
print('Final train accuracy:', history.history['accuracy'][-1])

## Evaluating & Predicting
`model.evaluate` runs the model over the full 10,000-image test set (data it never trained on) to get an honest measure of test accuracy - this is the number that best reflects how the model would perform on new, real-world digits. Beyond the aggregate accuracy, it's often useful to inspect individual predictions: `model.predict` on a single image returns a vector of 10 probabilities (one per digit class), and `np.argmax` picks out the class with the highest probability as the model's predicted digit. Comparing that predicted digit to the actual label for the same image is a quick sanity check that the model's predictions make sense, and is the same pattern you'd use to inspect misclassified examples when debugging a model.</cell id="cell-7">


In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)  # honest accuracy on unseen test images
print('Test accuracy:', test_acc)
import numpy as np
pred = model.predict(X_test[:1])  # predict on a single test image; returns 10 class probabilities
print('Predicted digit:', np.argmax(pred))  # index of the highest probability = predicted digit
print('Actual digit:', y_test[0])